In [10]:
from typing import Optional, Dict, Any, Tuple
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
import openai
from langchain_core.prompts import ChatPromptTemplate
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import Language
from langchain_openai import OpenAIEmbeddings  
# Embedding model
from langchain.document_loaders import TextLoader
from langchain_text_splitters import (
    Language,
    RecursiveCharacterTextSplitter,
)
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

In [11]:
# Setup model
# Load environment variables
load_dotenv()

# Retrieve API key
api_key = os.getenv("OPENAI_API_KEY")
# Ensure the API key is correctly set
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables")

# Initialize the ChatOpenAI model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=api_key  # Ensure you explicitly pass the API key
)

In [12]:
from typing import Optional, Dict, Any, Tuple
from langchain_openai import OpenAIEmbeddings   # ✅ dùng import mới, không warning
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate

# ====== CONFIG FAISS PATH ======
FAISS_DIR  = r"C:\Users\ASUS\anaconda3-project-code\wearable-2-sharing-non-compliance-evaluation\faiss_all_apps"
INDEX_NAME = "java_blocks_all"

# ====== 1) Load FAISS + Retrieve top-1 ======
def load_vectorstore(
    faiss_dir: str = FAISS_DIR,
    index_name: str = INDEX_NAME,
    api_key: str = api_key,
    embed_model: str = "text-embedding-3-small",
) -> FAISS:
    embeddings = OpenAIEmbeddings(model=embed_model, api_key=api_key)
    vs = FAISS.load_local(
        folder_path=faiss_dir,
        index_name=index_name,
        embeddings=embeddings,
        allow_dangerous_deserialization=True,
    )
    return vs

def retrieve_top1(code_block: str, vs: FAISS, k: int = 1):
    hits = vs.similarity_search_with_score(code_block, k=k)
    if not hits:
        return None
    return hits[0]  # (doc, score)


# ====== 2) RAG Generation: classify LEAK/NON-LEAK ======
def _build_context_from_doc(doc) -> str:
    md = doc.metadata or {}
    ctx_lines = [
        "[METADATA]",
        f"- doctype: {md.get('doctype')}",
        f"- app_name: {md.get('app_name')}",
        f"- category: {md.get('category')}",
        f"- rule_name: {md.get('rule_name')}" if md.get("rule_name") else "",
        f"- source_list: {', '.join(md.get('source_list', []))}" if md.get("source_list") else "",
        f"- sink_list: {', '.join(md.get('sink_list', []))}" if md.get("sink_list") else "",
        f"- source_path: {md.get('source') or md.get('txt_file')}",
        "",
        "[DOCUMENT_CONTENT]",
        doc.page_content[:2000],  # tránh quá dài
    ]
    return "\n".join([x for x in ctx_lines if x])


def rag_classify_code_block(
    code_block: str,
    vs: Optional[FAISS] = None,
    faiss_dir: str = FAISS_DIR,
    index_name: str = INDEX_NAME,
) -> Dict[str, Any]:
    """
    - Retrieve top-1 từ FAISS làm context
    - Nếu không có kết quả => NON-LEAK
    - Nếu có => dùng LLM phân loại; nếu thấy LEAK thì leak_type = category từ metadata
    """
    # Load vectorstore nếu chưa có
    if vs is None:
        vs = load_vectorstore(faiss_dir=faiss_dir, index_name=index_name)

    top1 = retrieve_top1(code_block, vs, k=1)
    if top1 is None:
        return {
            "verdict": "NON-LEAK",
            "reason": "No neighbor found in vector DB (top_k=1 returned nothing).",
            "leak_type": None,
            "matched_doc": None,
            "score": None,
        }

    doc, score = top1
    context = _build_context_from_doc(doc)
    category = (doc.metadata or {}).get("category")

    prompt = ChatPromptTemplate.from_template(
        """You are a static-analysis assistant. Determine if the INPUT Java code leaks sensitive data.
Use the CONTEXT (retrieved from a labeled vector DB) to make the decision.

Rules:
- If context strongly matches a known leak pattern, answer LEAK and report the leak type.
- Prefer the 'category' field from context as the leak type (e.g., 'PersonalInfo').
- If unsure or context is irrelevant, answer NON-LEAK.

Return ONLY a compact JSON with keys: verdict, leak_type, reason.

CONTEXT:
{context}

INPUT CODE:
```java
{code}
```"""
    )

    messages = prompt.format_messages(context=context, code=code_block)
    llm_output = llm.invoke(messages).content  # dùng ChatOpenAI đã init từ trước

    out = {
        "verdict": None,
        "leak_type": None,
        "reason": None,
        "matched_doc": {
            "category": category,
            "rule_name": (doc.metadata or {}).get("rule_name"),
            "doctype": (doc.metadata or {}).get("doctype"),
            "app_name": (doc.metadata or {}).get("app_name"),
            "source_path": (doc.metadata or {}).get("source") or (doc.metadata or {}).get("txt_file"),
            "preview": doc.page_content[:300],
        },
        "score": score,
        "raw_llm": llm_output,
    }

    # Thử parse JSON output của LLM
    try:
        import json
        js = json.loads(llm_output)
        out["verdict"]   = js.get("verdict")
        out["leak_type"] = js.get("leak_type") or category
        out["reason"]    = js.get("reason")
    except Exception:
        # fallback nếu LLM không trả đúng JSON
        out["verdict"]   = "LEAK"
        out["leak_type"] = category
        out["reason"]    = "Top-1 retrieved neighbor present; fallback classification."

    return out


In [13]:
test_code = """
class a24me.groupcal.utils.x0{
  public final java.lang.String G(){
    java.lang.String string=this.f9411E.getString("defSharedEmail","");
    kotlin.jvm.internal.n.e(string);
    return string;
  }
}
"""

res = rag_classify_code_block(test_code)
print(res)


{'verdict': 'LEAK', 'leak_type': 'PersonalInfo', 'reason': 'Top-1 retrieved neighbor present; fallback classification.', 'matched_doc': {'category': 'PersonalInfo', 'rule_name': 'PersonalInfo_Network', 'doctype': 'java_chunk', 'app_name': 'app.groupcal.www.apk', 'source_path': 'C:\\Users\\ASUS\\anaconda3-project-code\\wearable-2-sharing-non-compliance-evaluation\\source_sink_code_blocks\\app.groupcal.www.apk\\0\\4-PersonalInfo_Network-code-block-5.java', 'preview': 'class a24me.groupcal.utils.x0{\n  public final java.lang.String G(){\n    java.lang.String string=this.f9411E.getString("defSharedEmail","");\n    kotlin.jvm.internal.n.e(string);\n    return string;\n  }\n}'}, 'score': 0.07697039, 'raw_llm': '```json\n{\n  "verdict": "LEAK",\n  "leak_type": "PersonalInfo",\n  "reason": "The code retrieves a sensitive email address from SharedPreferences and could potentially expose it through an Intent."\n}\n```'}
